# NSGABLACK工具集详解（一）：核心基础工具

NSGABLACK框架提供了丰富的核心基础工具，为优化算法提供底层支持：

## Core模块核心工具

1. **Elite** - 精英个体管理
2. **Problems** - 问题定义与接口
3. **Solver** - 求解器基类
4. **历史解管理** - 优化过程追踪
5. **基础数据结构** - 通用数据容器

In [ ]:
# 导入必要的库
import numpy as np
import matplotlib.pyplot as plt
import time
from typing import List, Dict, Any, Optional, Tuple
from abc import ABC, abstractmethod
from collections import deque
import json
from dataclasses import dataclass, field

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("环境准备完成")

## 1. 精英管理器（Elite Manager）

In [ ]:
@dataclass
class Individual:
    """个体类"""
    position: np.ndarray
    fitness: float
    age: int = 0
    rank: int = 0
    crowding_distance: float = 0.0
    
    def __post_init__(self):
        self.position = np.asarray(self.position)
    
    def dominates(self, other: 'Individual') -> bool:
        """检查是否支配另一个个体"""
        return self.fitness < other.fitness

@dataclass
class MultiObjectiveIndividual:
    """多目标个体类"""
    position: np.ndarray
    objectives: np.ndarray
    age: int = 0
    rank: int = 0
    crowding_distance: float = 0.0
    
    def __post_init__(self):
        self.position = np.asarray(self.position)
        self.objectives = np.asarray(self.objectives)
    
    def dominates(self, other: 'MultiObjectiveIndividual') -> bool:
        """帕累托支配检查"""
        return (np.all(self.objectives <= other.objectives) and 
                np.any(self.objectives < other.objectives))

class EliteManager:
    """精英管理器 - 管理精英个体档案"""
    
    def __init__(self, max_size: int = 100, is_multi_objective: bool = False):
        """
        初始化精英管理器
        
        Args:
            max_size: 最大档案大小
            is_multi_objective: 是否多目标优化
        """
        self.max_size = max_size
        self.is_multi_objective = is_multi_objective
        self.archive = []
        self.history = []  # 历史最优
        
        print(f"创建精英管理器")
        print(f"  最大档案大小: {max_size}")
        print(f"  多目标模式: {is_multi_objective}")
    
    def update(self, population: List[Any]) -> List[Any]:
        """
        更新精英档案
        
        Args:
            population: 当前种群
        
        Returns:
            新加入的精英个体
        }
        return self._update_single_objective(population)
        """
    def _update_single_objective(self, population: List[Individual]) -> List[Individual]:
        """单目标精英更新"""
        new_elites = []
        
        for individual in population:
            # 检查是否应该加入档案
            should_add = True
            
            # 检查是否被档案中个体支配
            for elite in self.archive:
                if elite.dominates(individual):
                    should_add = False
                    break
            
            if should_add:
                # 移除被新个体支配的档案成员
                self.archive = [e for e in self.archive if not individual.dominates(e)]
                # 添加新精英
                new_elite = Individual(
                    position=individual.position.copy(),
                    fitness=individual.fitness
                )
                self.archive.append(new_elite)
                new_elites.append(new_elite)
        
        # 维护档案大小
        if len(self.archive) > self.max_size:
            self.archive.sort(key=lambda x: x.fitness)
            self.archive = self.archive[:self.max_size]
        
        # 更新历史最优
        if self.archive and (not self.history or self.archive[0].fitness < self.history[0].fitness):
            self.history = [Individual(
                position=self.archive[0].position.copy(),
                fitness=self.archive[0].fitness
            )]
        
        return new_elites
    
    def _update_multi_objective(self, population: List[MultiObjectiveIndividual]) -> List[MultiObjectiveIndividual]:
        """多目标精英更新"""
        new_elites = []
        
        for individual in population:
            # 检查是否应该加入档案
            should_add = True
            
            # 检查是否被档案中个体支配
            for elite in self.archive:
                if elite.dominates(individual):
                    should_add = False
                    break
            
            if should_add:
                # 移除被新个体支配的档案成员
                self.archive = [e for e in self.archive if not individual.dominates(e)]
                # 添加新精英
                new_elite = MultiObjectiveIndividual(
                    position=individual.position.copy(),
                    objectives=individual.objectives.copy()
                )
                self.archive.append(new_elite)
                new_elites.append(new_elite)
        
        # 维护档案大小（使用拥挤度距离）
        if len(self.archive) > self.max_size:
            self._crowding_distance_selection()
        
        return new_elites
    
    def _crowding_distance_selection(self):
        """基于拥挤度距离的选择"""
        if not self.archive:
            return
        
        # 计算拥挤度距离
        n_obj = len(self.archive[0].objectives)
        n_points = len(self.archive)
        
        # 初始化拥挤度
        for elite in self.archive:
            elite.crowding_distance = 0
        
        # 边界点设为无穷大
        if n_points > 0:
            self.archive[0].crowding_distance = np.inf
            self.archive[-1].crowding_distance = np.inf
        
        # 对每个目标计算拥挤度
        for i in range(n_obj):
            # 按第i个目标排序
            self.archive.sort(key=lambda x: x.objectives[i])
            
            # 目标值范围
            obj_range = self.archive[-1].objectives[i] - self.archive[0].objectives[i]
            
            if obj_range > 0:
                # 计算中间点的拥挤度
                for j in range(1, n_points - 1):
                    self.archive[j].crowding_distance += (
                        self.archive[j+1].objectives[i] - self.archive[j-1].objectives[i]
                    ) / obj_range
        
        # 选择拥挤度大的个体
        self.archive.sort(key=lambda x: x.crowding_distance, reverse=True)
        self.archive = self.archive[:self.max_size]
    
    def get_best(self, n: int = 1) -> List[Any]:
        """获取最优的n个个体"""
        if not self.archive:
            return []
        
        if self.is_multi_objective:
            # 多目标返回整个帕累托前沿
            return self.archive[:min(n, len(self.archive))]
        else:
            # 单目标按适应度排序
            sorted_archive = sorted(self.archive, key=lambda x: x.fitness)
            return sorted_archive[:n]
    
    def get_statistics(self) -> Dict[str, Any]:
        """获取档案统计信息"""
        if not self.archive:
            return {'size': 0}
        
        stats = {
            'size': len(self.archive),
            'max_size': self.max_size,
            'utilization': len(self.archive) / self.max_size * 100
        }
        
        if self.is_multi_objective:
            # 多目标统计
            objectives = np.array([ind.objectives for ind in self.archive])
            stats['objective_means'] = np.mean(objectives, axis=0)
            stats['objective_stds'] = np.std(objectives, axis=0)
            stats['hypervolume'] = self._calculate_hypervolume()
        else:
            # 单目标统计
            fitnesses = [ind.fitness for ind in self.archive]
            stats['best_fitness'] = min(fitnesses)
            stats['worst_fitness'] = max(fitnesses)
            stats['mean_fitness'] = np.mean(fitnesses)
            stats['std_fitness'] = np.std(fitnesses)
        
        return stats
    
    def _calculate_hypervolume(self, reference_point: Optional[np.ndarray] = None) -> float:
        """计算超体积（简化版）"""
        if not self.archive:
            return 0.0
        
        # 使用默认参考点
        if reference_point is None:
            objectives = np.array([ind.objectives for ind in self.archive])
            reference_point = np.max(objectives, axis=0) + 1
        
        # 简化的超体积计算（仅适用于二维）
        if len(reference_point) == 2:
            volume = 0
            sorted_archive = sorted(self.archive, key=lambda x: x.objectives[0])
            
            for elite in sorted_archive:
                if np.all(elite.objectives <= reference_point):
                    width = reference_point[0] - elite.objectives[0]
                    height = reference_point[1] - elite.objectives[1]
                    volume += width * height
            
            return volume
        
        return 0.0

# 测试精英管理器
print("测试精英管理器：")
print("=" * 50)

# 单目标测试
print("\n1. 单目标精英管理：")
single_elite_manager = EliteManager(max_size=5, is_multi_objective=False)

# 创建测试种群
population = [
    Individual(position=np.array([1.0]), fitness=10.0),
    Individual(position=np.array([2.0]), fitness=5.0),
    Individual(position=np.array([3.0]), fitness=8.0),
    Individual(position=np.array([4.0]), fitness=3.0),
    Individual(position=np.array([5.0]), fitness=7.0),
]

print(f"初始种群适应度: {[p.fitness for p in population]}")

# 更新精英档案
new_elites = single_elite_manager.update(population)
print(f"加入档案的新精英数: {len(new_elites)}")
print(f"档案大小: {len(single_elite_manager.archive)}")
print(f"档案精英适应度: {[e.fitness for e in single_elite_manager.archive]}")

# 添加更多个体
more_population = [
    Individual(position=np.array([6.0]), fitness=2.0),
    Individual(position=np.array([7.0]), fitness=4.0),
]

new_elites = single_elite_manager.update(more_population)
print(f"\n添加更多个体后：")
print(f"档案大小: {len(single_elite_manager.archive)}")
print(f"档案精英适应度: {[e.fitness for e in single_elite_manager.archive]}")

# 获取统计信息
stats = single_elite_manager.get_statistics()
print(f"\n档案统计：")
print(f"  最优适应度: {stats['best_fitness']:.4f}")
print(f"  平均适应度: {stats['mean_fitness']:.4f}")
print(f"  利用率: {stats['utilization']:.1f}%")

# 多目标测试
print("\n\n2. 多目标精英管理：")
multi_elite_manager = EliteManager(max_size=10, is_multi_objective=True)

# 创建多目标种群
multi_population = [
    MultiObjectiveIndividual(position=np.array([1, 1]), objectives=np.array([1, 5])),
    MultiObjectiveIndividual(position=np.array([2, 2]), objectives=np.array([2, 4])),
    MultiObjectiveIndividual(position=np.array([3, 3]), objectives=np.array([3, 3])),
    MultiObjectiveIndividual(position=np.array([4, 4]), objectives=np.array([4, 2])),
    MultiObjectiveIndividual(position=np.array([5, 5]), objectives=np.array([5, 1])),
]

print(f"初始种群目标值:")
for p in multi_population:
    print(f"  {p.objectives}")

# 更新档案
multi_elite_manager.update(multi_population)
print(f"\n档案大小: {len(multi_elite_manager.archive)}")
print(f"档案精英目标值:")
for e in multi_elite_manager.archive:
    print(f"  {e.objectives}")

# 可视化
plt.figure(figsize=(12, 5))

# 单目标精英演化
plt.subplot(1, 2, 1)
all_fitness = [p.fitness for p in population + more_population]
archive_fitness = [e.fitness for e in single_elite_manager.archive]
plt.scatter(range(len(all_fitness)), all_fitness, c='blue', s=50, 
           alpha=0.5, label='所有个体')
plt.scatter(range(len(archive_fitness)), archive_fitness, c='red', s=100, 
           marker='*', label='精英档案')
plt.xlabel('个体索引')
plt.ylabel('适应度')
plt.title('单目标精英档案')
plt.legend()
plt.grid(True, alpha=0.3)

# 多目标帕累托前沿
plt.subplot(1, 2, 2)
pop_objectives = np.array([p.objectives for p in multi_population])
archive_objectives = np.array([e.objectives for e in multi_elite_manager.archive])
plt.scatter(pop_objectives[:, 0], pop_objectives[:, 1], c='blue', s=50, 
           alpha=0.5, label='种群')
plt.scatter(archive_objectives[:, 0], archive_objectives[:, 1], c='red', s=100, 
           marker='*', label='帕累托前沿')
plt.xlabel('目标1')
plt.ylabel('目标2')
plt.title('多目标帕累托前沿')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. 问题定义（Problem Definition）

In [ ]:
class Problem(ABC):
    """问题基类"""
    
    def __init__(self, name: str, n_var: int, bounds: List[Tuple[float, float]]):
        """
        初始化问题
        
        Args:
            name: 问题名称
            n_var: 变量维度
            bounds: 变量边界 [(low1, high1), ...]
        """
        self.name = name
        self.n_var = n_var
        self.bounds = bounds
        self.n_constraints = 0
        self.optimal_value = None
        self.optimal_solution = None
    
    @abstractmethod
    def evaluate(self, x: np.ndarray) -> float:
        """评估单个解"""
        pass
    
    def evaluate_batch(self, X: np.ndarray) -> np.ndarray:
        """批量评估"""
        return np.array([self.evaluate(x) for x in X])
    
    def is_feasible(self, x: np.ndarray) -> bool:
        """检查解是否可行"""
        return True  # 默认无约束
    
    def repair(self, x: np.ndarray) -> np.ndarray:
        """修复不可行解"""
        # 简单的边界修复
        repaired = x.copy()
        for i, (low, high) in enumerate(self.bounds):
            repaired[i] = np.clip(repaired[i], low, high)
        return repaired

class MultiObjectiveProblem(Problem):
    """多目标问题基类"""
    
    def __init__(self, name: str, n_var: int, bounds: List[Tuple[float, float]], 
                 n_objectives: int):
        super().__init__(name, n_var, bounds)
        self.n_objectives = n_objectives
    
    @abstractmethod
    def evaluate_objectives(self, x: np.ndarray) -> np.ndarray:
        """评估所有目标"""
        pass
    
    def evaluate(self, x: np.ndarray) -> float:
        """默认返回第一个目标（保持兼容性）"""
        objectives = self.evaluate_objectives(x)
        return objectives[0]

# 具体问题实现
class SphereProblem(Problem):
    """Sphere函数 - 最简单的测试函数"""
    
    def __init__(self, n_var: int = 10, bounds: List[Tuple[float, float]] = None):
        if bounds is None:
            bounds = [(-100, 100)] * n_var
        super().__init__("Sphere", n_var, bounds)
        self.optimal_value = 0.0
        self.optimal_solution = np.zeros(n_var)
    
    def evaluate(self, x: np.ndarray) -> float:
        return np.sum(x**2)

class RastriginProblem(Problem):
    """Rastrigin函数 - 多峰测试函数"""
    
    def __init__(self, n_var: int = 10, bounds: List[Tuple[float, float]] = None):
        if bounds is None:
            bounds = [(-5.12, 5.12)] * n_var
        super().__init__("Rastrigin", n_var, bounds)
        self.optimal_value = 0.0
        self.optimal_solution = np.zeros(n_var)
    
    def evaluate(self, x: np.ndarray) -> float:
        A = 10
        return A * len(x) + np.sum(x**2 - A * np.cos(2 * np.pi * x))

class ZDT1Problem(MultiObjectiveProblem):
    """ZDT1 - 经典双目标测试问题"""
    
    def __init__(self, n_var: int = 30):
        bounds = [(0, 1)] * n_var
        super().__init__("ZDT1", n_var, bounds, n_objectives=2)
    
    def evaluate_objectives(self, x: np.ndarray) -> np.ndarray:
        f1 = x[0]
        g = 1 + 9 * np.sum(x[1:]) / (len(x) - 1)
        f2 = g * (1 - np.sqrt(f1 / g))
        return np.array([f1, f2])

class ConstrainedSphere(Problem):
    """带约束的Sphere问题"""
    
    def __init__(self, n_var: int = 2):
        bounds = [(-2, 2)] * n_var
        super().__init__("ConstrainedSphere", n_var, bounds)
        self.n_constraints = 1
    
    def evaluate(self, x: np.ndarray) -> float:
        return np.sum(x**2)
    
    def is_feasible(self, x: np.ndarray) -> bool:
        # 约束: x1 + x2 >= 1
        return np.sum(x) >= 1
    
    def constraint_violation(self, x: np.ndarray) -> float:
        """计算约束违反程度"""
        return max(0, 1 - np.sum(x))

# 测试问题定义
print("测试问题定义：")
print("=" * 50)

# 创建测试问题
sphere = SphereProblem(n_var=2)
rastrigin = RastriginProblem(n_var=2)
zdt1 = ZDT1Problem(n_var=2)
constrained = ConstrainedSphere(n_var=2)

# 测试评估
test_points = [
    np.array([0.0, 0.0]),
    np.array([1.0, 1.0]),
    np.array([-1.0, 2.0])
]

print("\n1. Sphere函数评估：")
for i, x in enumerate(test_points):
    fitness = sphere.evaluate(x)
    print(f"  点{i+1}: x={x}, f(x)={fitness:.4f}")

print("\n2. Rastrigin函数评估：")
for i, x in enumerate(test_points):
    fitness = rastrigin.evaluate(x)
    print(f"  点{i+1}: x={x}, f(x)={fitness:.4f}")

print("\n3. ZDT1多目标评估：")
for i, x in enumerate(test_points):
    objectives = zdt1.evaluate_objectives(x)
    print(f"  点{i+1}: x={x}, f1={objectives[0]:.4f}, f2={objectives[1]:.4f}")

print("\n4. 约束问题评估：")
for i, x in enumerate(test_points):
    fitness = constrained.evaluate(x)
    feasible = constrained.is_feasible(x)
    violation = constrained.constraint_violation(x)
    print(f"  点{i+1}: x={x}, f={fitness:.4f}, 可行={feasible}, 违反={violation:.4f}")

# 可视化问题
plt.figure(figsize=(15, 10))

# 创建网格
x = np.linspace(-2, 2, 100)
y = np.linspace(-2, 2, 100)
X, Y = np.meshgrid(x, y)

# Sphere函数
plt.subplot(2, 3, 1)
Z = np.zeros_like(X)
for i in range(100):
    for j in range(100):
        Z[i, j] = sphere.evaluate(np.array([X[i, j], Y[i, j]]))
contour = plt.contourf(X, Y, Z, levels=20)
plt.colorbar(contour)
plt.title('Sphere函数')
plt.xlabel('x1')
plt.ylabel('x2')

# Rastrigin函数
plt.subplot(2, 3, 2)
Z = np.zeros_like(X)
for i in range(100):
    for j in range(100):
        Z[i, j] = rastrigin.evaluate(np.array([X[i, j], Y[i, j]]))
contour = plt.contourf(X, Y, Z, levels=20)
plt.colorbar(contour)
plt.title('Rastrigin函数')
plt.xlabel('x1')
plt.ylabel('x2')

# ZDT1目标1
plt.subplot(2, 3, 3)
Z = np.zeros_like(X)
for i in range(100):
    for j in range(100):
        objectives = zdt1.evaluate_objectives(np.array([X[i, j], Y[i, j]]))
        Z[i, j] = objectives[0]
contour = plt.contourf(X, Y, Z, levels=20)
plt.colorbar(contour)
plt.title('ZDT1 - 目标1')
plt.xlabel('x1')
plt.ylabel('x2')

# ZDT1目标2
plt.subplot(2, 3, 4)
Z = np.zeros_like(X)
for i in range(100):
    for j in range(100):
        objectives = zdt1.evaluate_objectives(np.array([X[i, j], Y[i, j]]))
        Z[i, j] = objectives[1]
contour = plt.contourf(X, Y, Z, levels=20)
plt.colorbar(contour)
plt.title('ZDT1 - 目标2')
plt.xlabel('x1')
plt.ylabel('x2')

# 约束问题
plt.subplot(2, 3, 5)
Z = np.zeros_like(X)
feasible_mask = np.zeros_like(X, dtype=bool)
for i in range(100):
    for j in range(100):
        point = np.array([X[i, j], Y[i, j]])
        Z[i, j] = constrained.evaluate(point)
        feasible_mask[i, j] = constrained.is_feasible(point)

# 绘制不可行区域
plt.contourf(X, Y, ~feasible_mask, levels=[0.5, 1], colors=['red'], alpha=0.3)
contour = plt.contour(X, Y, Z, levels=10)
plt.clabel(contour, inline=True, fontsize=8)
# 绘制约束线
x_line = np.linspace(-2, 3, 100)
y_line = 1 - x_line
plt.plot(x_line, y_line, 'r--', linewidth=2, label='约束边界')
plt.title('约束Sphere问题')
plt.xlabel('x1')
plt.ylabel('x2')
plt.legend()
plt.xlim(-2, 2)
plt.ylim(-2, 2)

# ZDT1帕累托前沿
plt.subplot(2, 3, 6)
# 生成帕累托前沿
f1_values = np.linspace(0, 1, 100)
f2_values = 1 - np.sqrt(f1_values)
plt.plot(f1_values, f2_values, 'b-', linewidth=2, label='真实帕累托前沿')

# 随机采样点
np.random.seed(42)
for _ in range(50):
    x = np.random.rand(2)
    objectives = zdt1.evaluate_objectives(x)
    plt.scatter(objectives[0], objectives[1], c='red', s=20, alpha=0.5)

plt.xlabel('目标1')
plt.ylabel('目标2')
plt.title('ZDT1目标空间')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. 求解器基类（Solver Base）

In [ ]:
class Solver(ABC):
    """求解器基类"""
    
    def __init__(self, name: str, **kwargs):
        """
        初始化求解器
        
        Args:
            name: 求解器名称
            **kwargs: 其他参数
        """
        self.name = name
        self.parameters = kwargs
        
        # 运行状态
        self.is_initialized = False
        self.is_running = False
        self.is_finished = False
        
        # 结果存储
        self.best_solution = None
        self.best_fitness = float('inf')
        self.history = []
        self.convergence_data = {}
        
        # 统计信息
        self.n_evaluations = 0
        self.elapsed_time = 0
        self.start_time = None
        
        print(f"创建求解器: {name}")
    
    @abstractmethod
    def _initialize(self, problem: Problem) -> None:
        """初始化求解器"""
        pass
    
    @abstractmethod
    def _step(self, problem: Problem) -> None:
        """执行一步优化"""
        pass
    
    def solve(self, problem: Problem, max_evaluations: int = 10000, 
              max_time: float = None, verbose: bool = True) -> Dict[str, Any]:
        """
        求解问题
        
        Args:
            problem: 优化问题
            max_evaluations: 最大评估次数
            max_time: 最大运行时间（秒）
            verbose: 是否输出详细信息
        
        Returns:
            优化结果
        """
        print(f"\n开始求解: {problem.name}")
        print(f"求解器: {self.name}")
        print(f"最大评估次数: {max_evaluations}")
        
        # 初始化
        self._initialize(problem)
        self.is_initialized = True
        self.is_running = True
        self.start_time = time.time()
        self.n_evaluations = 0
        
        # 主循环
        while (self.n_evaluations < max_evaluations and 
               self.is_running and
               (max_time is None or time.time() - self.start_time < max_time)):
            
            # 执行一步
            self._step(problem)
            
            # 记录历史
            if len(self.history) == 0 or self.best_fitness != self.history[-1]:
                self.history.append(self.best_fitness)
            
            # 输出进度
            if verbose and self.n_evaluations % 100 == 0:
                print(f"  评估 {self.n_evaluations}: 最优值 = {self.best_fitness:.6f}")
        
        # 结束
        self.is_running = False
        self.is_finished = True
        self.elapsed_time = time.time() - self.start_time
        
        print(f"\n求解完成！")
        print(f"最优适应度: {self.best_fitness:.6f}")
        print(f"评估次数: {self.n_evaluations}")
        print(f"运行时间: {self.elapsed_time:.2f}秒")
        
        # 返回结果
        return self.get_results()
    
    def get_results(self) -> Dict[str, Any]:
        """获取求解结果"""
        return {
            'best_solution': self.best_solution,
            'best_fitness': self.best_fitness,
            'history': self.history.copy(),
            'n_evaluations': self.n_evaluations,
            'elapsed_time': self.elapsed_time,
            'convergence_data': self.convergence_data.copy()
        }
    
    def reset(self) -> None:
        """重置求解器"""
        self.is_initialized = False
        self.is_running = False
        self.is_finished = False
        self.best_solution = None
        self.best_fitness = float('inf')
        self.history = []
        self.convergence_data = {}
        self.n_evaluations = 0
        self.elapsed_time = 0
        self.start_time = None

# 具体求解器实现
class RandomSearchSolver(Solver):
    """随机搜索求解器"""
    
    def __init__(self, seed: int = None):
        super().__init__("RandomSearch", seed=seed)
        self.seed = seed
        if seed is not None:
            np.random.seed(seed)
    
    def _initialize(self, problem: Problem) -> None:
        # 不需要特殊初始化
        pass
    
    def _step(self, problem: Problem) -> None:
        # 生成随机解
        x = np.array([
            np.random.uniform(low, high) 
            for low, high in problem.bounds
        ])
        
        # 评估
        fitness = problem.evaluate(x)
        self.n_evaluations += 1
        
        # 更新最优
        if fitness < self.best_fitness:
            self.best_fitness = fitness
            self.best_solution = x.copy()

class LocalSearchSolver(Solver):
    """局部搜索求解器"""
    
    def __init__(self, step_size: float = 0.1, seed: int = None):
        super().__init__("LocalSearch", step_size=step_size, seed=seed)
        self.step_size = step_size
        self.seed = seed
        if seed is not None:
            np.random.seed(seed)
        self.current_solution = None
        self.current_fitness = float('inf')
    
    def _initialize(self, problem: Problem) -> None:
        # 随机初始解
        self.current_solution = np.array([
            np.random.uniform(low, high) 
            for low, high in problem.bounds
        ])
        self.current_fitness = problem.evaluate(self.current_solution)
        self.n_evaluations += 1
        
        self.best_solution = self.current_solution.copy()
        self.best_fitness = self.current_fitness
    
    def _step(self, problem: Problem) -> None:
        improved = False
        
        # 尝试每个维度的邻域
        for i in range(problem.n_var):
            # 正方向
            new_solution = self.current_solution.copy()
            new_solution[i] += self.step_size * (problem.bounds[i][1] - problem.bounds[i][0])
            new_solution[i] = min(new_solution[i], problem.bounds[i][1])
            
            fitness = problem.evaluate(new_solution)
            self.n_evaluations += 1
            
            if fitness < self.current_fitness:
                self.current_solution = new_solution
                self.current_fitness = fitness
                improved = True
                continue
            
            # 负方向
            new_solution = self.current_solution.copy()
            new_solution[i] -= self.step_size * (problem.bounds[i][1] - problem.bounds[i][0])
            new_solution[i] = max(new_solution[i], problem.bounds[i][0])
            
            fitness = problem.evaluate(new_solution)
            self.n_evaluations += 1
            
            if fitness < self.current_fitness:
                self.current_solution = new_solution
                self.current_fitness = fitness
                improved = True
        
        # 更新全局最优
        if self.current_fitness < self.best_fitness:
            self.best_fitness = self.current_fitness
            self.best_solution = self.current_solution.copy()

# 测试求解器
print("\n测试求解器：")
print("=" * 50)

# 创建测试问题
test_problem = SphereProblem(n_var=2)

# 测试随机搜索
print("\n1. 随机搜索求解器：")
random_solver = RandomSearchSolver(seed=42)
random_results = random_solver.solve(test_problem, max_evaluations=1000)

# 测试局部搜索
print("\n2. 局部搜索求解器：")
local_solver = LocalSearchSolver(step_size=0.1, seed=42)
local_results = local_solver.solve(test_problem, max_evaluations=1000)

# 可视化结果对比
plt.figure(figsize=(12, 5))

# 收敛曲线
plt.subplot(1, 2, 1)
plt.plot(random_results['history'], 'b-', label='随机搜索', linewidth=2)
plt.plot(local_results['history'], 'r-', label='局部搜索', linewidth=2)
plt.xlabel('改进次数')
plt.ylabel('最优适应度')
plt.title('收敛曲线对比')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')

# 性能对比
plt.subplot(1, 2, 2)
solvers = ['随机搜索', '局部搜索']
fitnesses = [random_results['best_fitness'], local_results['best_fitness']]
times = [random_results['elapsed_time'], local_results['elapsed_time']]

x = np.arange(len(solvers))
width = 0.35

plt.bar(x - width/2, fitnesses, width, label='最优适应度', alpha=0.7)
plt.bar(x + width/2, times, width, label='运行时间(秒)', alpha=0.7)

plt.xlabel('求解器')
plt.ylabel('值')
plt.title('性能对比')
plt.xticks(x, solvers)
plt.legend()
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 4. 历史解管理器（Solution History Manager）

In [ ]:
@dataclass
class HistoryEntry:
    """历史记录条目"""
    generation: int
    solution: np.ndarray
    fitness: float
    timestamp: float = field(default_factory=time.time)
    metadata: Dict[str, Any] = field(default_factory=dict)

class SolutionHistoryManager:
    """解历史管理器"""
    
    def __init__(self, max_entries: int = 10000):
        """
        初始化历史管理器
        
        Args:
            max_entries: 最大历史记录数
        """
        self.max_entries = max_entries
        self.entries: List[HistoryEntry] = []
        self.best_history: List[HistoryEntry] = []  # 历史最优
        self.diversity_history: List[float] = []  # 多样性历史
        
        print(f"创建解历史管理器")
        print(f"  最大记录数: {max_entries}")
    
    def add_entry(self, generation: int, solution: np.ndarray, 
                 fitness: float, metadata: Dict[str, Any] = None) -> None:
        """
        添加历史记录
        
        Args:
            generation: 代数
            solution: 解
            fitness: 适应度
            metadata: 元数据
        """
        entry = HistoryEntry(
            generation=generation,
            solution=np.asarray(solution).copy(),
            fitness=fitness,
            metadata=metadata or {}
        )
        
        self.entries.append(entry)
        
        # 维护最大记录数
        if len(self.entries) > self.max_entries:
            self.entries.pop(0)
        
        # 更新最优历史
        self._update_best_history(entry)
    
    def _update_best_history(self, entry: HistoryEntry) -> None:
        """更新最优历史"""
        if not self.best_history or entry.fitness < self.best_history[-1].fitness:
            # 创建新的最优记录
            best_entry = HistoryEntry(
                generation=entry.generation,
                solution=entry.solution.copy(),
                fitness=entry.fitness,
                timestamp=entry.timestamp
            )
            self.best_history.append(best_entry)
    
    def get_best_solutions(self, n: int = 10) -> List[HistoryEntry]:
        """获取n个最优解"""
        if not self.entries:
            return []
        
        # 按适应度排序
        sorted_entries = sorted(self.entries, key=lambda x: x.fitness)
        return sorted_entries[:n]
    
    def get_generation_range(self, start_gen: int, end_gen: int) -> List[HistoryEntry]:
        """获取指定代数范围的记录"""
        return [e for e in self.entries if start_gen <= e.generation <= end_gen]
    
    def calculate_diversity(self, window_size: int = 100) -> float:
        """计算当前种群的多样性"""
        if len(self.entries) < 2:
            return 0.0
        
        # 获取最近的记录
        recent_entries = self.entries[-window_size:]
        
        # 计算解之间的平均距离
        solutions = np.array([e.solution for e in recent_entries])
        
        if len(solutions) < 2:
            return 0.0
        
        # 计算所有点对距离
        distances = []
        for i in range(len(solutions)):
            for j in range(i + 1, len(solutions)):
                dist = np.linalg.norm(solutions[i] - solutions[j])
                distances.append(dist)
        
        diversity = np.mean(distances) if distances else 0.0
        self.diversity_history.append(diversity)
        
        return diversity
    
    def detect_stagnation(self, window_size: int = 100, 
                         threshold: float = 1e-6) -> bool:
        """检测优化是否停滞"""
        if len(self.best_history) < window_size:
            return False
        
        # 获取最近的最优历史
        recent_best = self.best_history[-window_size:]
        fitnesses = [e.fitness for e in recent_best]
        
        # 计算改进
        improvement = fitnesses[0] - fitnesses[-1]
        
        return improvement < threshold
    
    def get_statistics(self) -> Dict[str, Any]:
        """获取统计信息"""
        if not self.entries:
            return {'total_entries': 0}
        
        fitnesses = [e.fitness for e in self.entries]
        generations = [e.generation for e in self.entries]
        
        stats = {
            'total_entries': len(self.entries),
            'generation_range': (min(generations), max(generations)),
            'best_fitness': min(fitnesses),
            'worst_fitness': max(fitnesses),
            'mean_fitness': np.mean(fitnesses),
            'fitness_std': np.std(fitnesses),
            'n_improvements': len(self.best_history),
            'current_diversity': self.diversity_history[-1] if self.diversity_history else 0
        }
        
        return stats
    
    def export_history(self, filename: str) -> None:
        """导出历史到文件"""
        export_data = {
            'entries': [
                {
                    'generation': e.generation,
                    'solution': e.solution.tolist(),
                    'fitness': e.fitness,
                    'timestamp': e.timestamp,
                    'metadata': e.metadata
                } for e in self.entries
            ],
            'statistics': self.get_statistics()
        }
        
        with open(filename, 'w') as f:
            json.dump(export_data, f, indent=2)
        
        print(f"历史已导出到: {filename}")

# 测试历史管理器
print("\n测试解历史管理器：")
print("=" * 50)

# 创建历史管理器
history_manager = SolutionHistoryManager(max_entries=1000)

# 模拟优化过程
np.random.seed(42)
current_best = float('inf')

print("\n模拟优化过程...")
for gen in range(100):
    # 生成随机解
    solution = np.random.randn(2)
    fitness = np.sum(solution**2) + np.random.normal(0, 0.1)
    
    # 添加到历史
    history_manager.add_entry(
        generation=gen,
        solution=solution,
        fitness=fitness,
        metadata={'noise': np.random.normal(0, 0.01)}
    )
    
    # 每20代输出统计
    if (gen + 1) % 20 == 0:
        diversity = history_manager.calculate_diversity()
        stagnated = history_manager.detect_stagnation()
        stats = history_manager.get_statistics()
        print(f"\n代数 {gen+1}:")
        print(f"  当前最优: {stats['best_fitness']:.4f}")
        print(f"  多样性: {diversity:.4f}")
        print(f"  是否停滞: {stagnated}")
        print(f"  改进次数: {stats['n_improvements']}")

# 获取最优解
print("\n\n获取10个最优解：")
best_solutions = history_manager.get_best_solutions(10)
for i, entry in enumerate(best_solutions):
    print(f"  第{i+1}: 代数={entry.generation}, 适应度={entry.fitness:.4f}")

# 可视化历史
plt.figure(figsize=(15, 10))

# 适应度历史
plt.subplot(2, 3, 1)
generations = [e.generation for e in history_manager.entries]
fitnesses = [e.fitness for e in history_manager.entries]
plt.plot(generations, fitnesses, 'b-', alpha=0.5, linewidth=1)
plt.scatter(generations, fitnesses, s=10, alpha=0.5)
plt.xlabel('代数')
plt.ylabel('适应度')
plt.title('完整适应度历史')
plt.grid(True, alpha=0.3)

# 最优历史
plt.subplot(2, 3, 2)
best_gen = [e.generation for e in history_manager.best_history]
best_fit = [e.fitness for e in history_manager.best_history]
plt.plot(best_gen, best_fit, 'r-o', linewidth=2)
plt.xlabel('改进次数')
plt.ylabel('最优适应度')
plt.title('最优适应度改进历史')
plt.grid(True, alpha=0.3)

# 多样性历史
plt.subplot(2, 3, 3)
if history_manager.diversity_history:
    plt.plot(history_manager.diversity_history, 'g-', linewidth=2)
    plt.xlabel('时间步')
    plt.ylabel('多样性')
    plt.title('种群多样性变化')
    plt.grid(True, alpha=0.3)

# 适应度分布
plt.subplot(2, 3, 4)
plt.hist(fitnesses, bins=30, alpha=0.7, edgecolor='black')
plt.xlabel('适应度')
plt.ylabel('频次')
plt.title('适应度分布')
plt.grid(True, alpha=0.3)

# 解空间分布（二维）
plt.subplot(2, 3, 5)
solutions = np.array([e.solution for e in history_manager.entries])
if solutions.shape[1] == 2:
    plt.scatter(solutions[:, 0], solutions[:, 1], c=fitnesses, 
               cmap='viridis', alpha=0.6)
    plt.colorbar(label='适应度')
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.title('解空间分布')
    plt.grid(True, alpha=0.3)

# 改进幅度
plt.subplot(2, 3, 6)
if len(best_fit) > 1:
    improvements = [best_fit[i-1] - best_fit[i] for i in range(1, len(best_fit))]
    plt.bar(range(len(improvements)), improvements, alpha=0.7)
    plt.xlabel('改进次数')
    plt.ylabel('改进幅度')
    plt.title('每次改进的幅度')
    plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# 导出历史
history_manager.export_history('optimization_history.json')

# 最终统计
final_stats = history_manager.get_statistics()
print("\n最终统计：")
for key, value in final_stats.items():
    print(f"  {key}: {value}")

## 5. 基础数据结构（Basic Data Structures）

In [ ]:
from enum import Enum
from typing import Union, List, Dict, Set, Tuple

class SolutionStatus(Enum):
    """解的状态枚举"""
    PENDING = "pending"
    EVALUATED = "evaluated"
    INFEASIBLE = "infeasible"
    DOMINATED = "dominated"
    NON_DOMINATED = "non_dominated"
    ELITE = "elite"

@dataclass
class Solution:
    """通用解类"""
    id: int
    variables: np.ndarray
    objectives: np.ndarray
    constraints: np.ndarray = None
    status: SolutionStatus = SolutionStatus.PENDING
    rank: int = -1
    crowding_distance: float = 0.0
    fitness: float = float('inf')
    metadata: Dict[str, Any] = field(default_factory=dict)
    
    def __post_init__(self):
        self.variables = np.asarray(self.variables)
        self.objectives = np.asarray(self.objectives)
        if self.constraints is not None:
            self.constraints = np.asarray(self.constraints)
    
    def is_feasible(self) -> bool:
        """检查解是否可行"""
        if self.constraints is None:
            return True
        return np.all(self.constraints <= 0)
    
    def dominates(self, other: 'Solution') -> bool:
        """检查是否支配另一个解"""
        # 两个解都必须可行
        if not self.is_feasible() or not other.is_feasible():
            return False
        
        # 检查支配关系
        return (np.all(self.objectives <= other.objectives) and 
                np.any(self.objectives < other.objectives))

class Population:
    """种群类"""
    
    def __init__(self, max_size: int = None):
        """
        初始化种群
        
        Args:
            max_size: 最大种群大小
        """
        self.max_size = max_size
        self.individuals: List[Solution] = []
        self.next_id = 0
    
    def add(self, variables: np.ndarray, objectives: np.ndarray, 
            constraints: np.ndarray = None, **metadata) -> Solution:
        """
        添加解到种群
        
        Args:
            variables: 变量值
            objectives: 目标值
            constraints: 约束值
            **metadata: 其他元数据
        
        Returns:
            添加的解
        """
        solution = Solution(
            id=self.next_id,
            variables=np.asarray(variables),
            objectives=np.asarray(objectives),
            constraints=np.asarray(constraints) if constraints is not None else None,
            metadata=metadata
        )
        
        self.individuals.append(solution)
        self.next_id += 1
        
        # 维护最大大小
        if self.max_size and len(self.individuals) > self.max_size:
            self.individuals.pop(0)
        
        return solution
    
    def get_feasible(self) -> List[Solution]:
        """获取所有可行解"""
        return [ind for ind in self.individuals if ind.is_feasible()]
    
    def get_non_dominated(self) -> List[Solution]:
        """获取非支配解"""
        feasible = self.get_feasible()
        non_dominated = []
        
        for ind in feasible:
            is_dominated = False
            for other in feasible:
                if other.id != ind.id and other.dominates(ind):
                    is_dominated = True
                    break
            
            if not is_dominated:
                non_dominated.append(ind)
        
        return non_dominated
    
    def get_best(self, n: int = 1) -> List[Solution]:
        """获取最优的n个解（单目标）"""
        feasible = self.get_feasible()
        if not feasible:
            return []
        
        # 假设第一个目标是要优化的
        sorted_ind = sorted(feasible, key=lambda x: x.objectives[0])
        return sorted_ind[:n]
    
    def calculate_statistics(self) -> Dict[str, Any]:
        """计算种群统计信息"""
        if not self.individuals:
            return {'size': 0}
        
        feasible = self.get_feasible()
        non_dominated = self.get_non_dominated()
        
        stats = {
            'size': len(self.individuals),
            'feasible_count': len(feasible),
            'infeasible_count': len(self.individuals) - len(feasible),
            'non_dominated_count': len(non_dominated),
            'feasibility_ratio': len(feasible) / len(self.individuals) if self.individuals else 0
        }
        
        # 目标统计
        if feasible:
            objectives = np.array([ind.objectives for ind in feasible])
            stats.update({
                'objective_means': np.mean(objectives, axis=0),
                'objective_stds': np.std(objectives, axis=0),
                'objective_mins': np.min(objectives, axis=0),
                'objective_maxs': np.max(objectives, axis=0)
            })
        
        return stats
    
    def clear(self) -> None:
        """清空种群"""
        self.individuals.clear()
        self.next_id = 0

# 测试基础数据结构
print("\n测试基础数据结构：")
print("=" * 50)

# 创建种群
population = Population(max_size=50)

# 添加一些解
print("\n1. 添加解到种群：")
for i in range(20):
    x = np.random.randn(2)
    f1 = np.sum(x**2)  # 目标1：最小化
    f2 = np.sum((x - 2)**2)  # 目标2：最小化
    
    # 添加约束（示例）
    constraint = np.sum(x) - 1  # 约束: sum(x) <= 1
    
    solution = population.add(
        variables=x,
        objectives=np.array([f1, f2]),
        constraints=np.array([constraint]),
        generation=i
    )
    
    solution.status = SolutionStatus.EVALUATED

# 获取统计信息
stats = population.calculate_statistics()
print(f"\n种群统计：")
print(f"  种群大小: {stats['size']}")
print(f"  可行解数: {stats['feasible_count']}")
print(f"  非支配解数: {stats['non_dominated_count']}")
print(f"  可行性比例: {stats['feasibility_ratio']:.2%}")
if 'objective_means' in stats:
    print(f"  目标均值: {stats['objective_means']}")

# 获取非支配解
print("\n2. 非支配解：")
pareto_solutions = population.get_non_dominated()
print(f"找到 {len(pareto_solutions)} 个非支配解")
for i, sol in enumerate(pareto_solutions[:5]):
    print(f"  解{i+1}: 目标={sol.objectives}, 可行={sol.is_feasible()}")

# 可视化
plt.figure(figsize=(15, 5))

# 种群在目标空间
plt.subplot(1, 3, 1)
all_objectives = np.array([ind.objectives for ind in population.individuals])
feasible_objectives = np.array([ind.objectives for ind in population.get_feasible()])
pareto_objectives = np.array([ind.objectives for ind in pareto_solutions])

plt.scatter(all_objectives[:, 0], all_objectives[:, 1], 
           c='gray', s=30, alpha=0.5, label='所有解')
plt.scatter(feasible_objectives[:, 0], feasible_objectives[:, 1], 
           c='blue', s=50, alpha=0.7, label='可行解')
plt.scatter(pareto_objectives[:, 0], pareto_objectives[:, 1], 
           c='red', s=100, marker='*', label='非支配解')
plt.xlabel('目标1')
plt.ylabel('目标2')
plt.title('种群在目标空间的分布')
plt.legend()
plt.grid(True, alpha=0.3)

# 种群在决策空间
plt.subplot(1, 3, 2)
all_variables = np.array([ind.variables for ind in population.individuals])
feasible_variables = np.array([ind.variables for ind in population.get_feasible()])
pareto_variables = np.array([ind.variables for ind in pareto_solutions])

plt.scatter(all_variables[:, 0], all_variables[:, 1], 
           c='gray', s=30, alpha=0.5, label='所有解')
plt.scatter(feasible_variables[:, 0], feasible_variables[:, 1], 
           c='blue', s=50, alpha=0.7, label='可行解')
plt.scatter(pareto_variables[:, 0], pareto_variables[:, 1], 
           c='red', s=100, marker='*', label='非支配解')
# 绘制约束边界
x_line = np.linspace(-3, 4, 100)
y_line = 1 - x_line
plt.plot(x_line, y_line, 'r--', linewidth=2, label='约束边界')
plt.xlabel('x1')
plt.ylabel('x2')
plt.title('种群在决策空间的分布')
plt.legend()
plt.grid(True, alpha=0.3)

# 统计信息
plt.subplot(1, 3, 3)
categories = ['总体', '可行', '不可行', '非支配']
counts = [
    stats['size'],
    stats['feasible_count'],
    stats['infeasible_count'],
    stats['non_dominated_count']
]
colors = ['gray', 'blue', 'orange', 'red']
bars = plt.bar(categories, counts, alpha=0.7, color=colors)
plt.ylabel('数量')
plt.title('解的分类统计')
plt.grid(True, alpha=0.3, axis='y')

# 添加数值标签
for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(), 
             str(count), ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 总结

本节介绍了NSGABLACK框架的核心基础工具：

### 1. 精英管理器（EliteManager）
- 维护精英个体档案
- 支持单目标和多目标
- 自动更新和大小控制
- 拥挤度距离选择

### 2. 问题定义（Problem）
- 统一的问题接口
- 单目标和多目标基类
- 约束处理支持
- 解修复机制

### 3. 求解器基类（Solver）
- 标准化求解流程
- 状态管理和结果记录
- 可扩展的步骤接口
- 统一的参数系统

### 4. 历史解管理器
- 完整的优化历史记录
- 多样性监测
- 停滞检测
- 统计分析和导出

### 5. 基础数据结构
- 通用的解表示
- 种群管理
- 非支配关系处理
- 约束支持

这些核心工具为NSGABLACK框架提供了坚实的基础，使得算法实现更加规范和高效。